# Spotify Dataset — Exploratory Data Analysis

BANA409 Assignment 4, Part 1. This notebook documents data shape, types, missingness, distributions and candidate relationships before the dashboard design.

**Source:** https://www.kaggle.com/datasets/vedauv/global-spotify-hits-and-audio-dynamics-20182026


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

root = Path.cwd()
metadata = pd.read_csv(root / 'track_metadata.csv', parse_dates=['release_date'])
audio = pd.read_csv(root / 'audio_features.csv')
charts = pd.read_csv(root / 'chart_longevity_panel.csv')
df = metadata.merge(audio, on='track_id', validate='one_to_one').merge(charts, on='track_id', validate='one_to_one')
print('Source shapes:', metadata.shape, audio.shape, charts.shape)
print('Merged shape:', df.shape)
df.head()


## 1. Data types and missing values

Confirm that each track is unique and that the three source tables join one-to-one.


In [ ]:
display(pd.DataFrame({'dtype':df.dtypes.astype(str), 'missing':df.isna().sum(), 'missing_pct':df.isna().mean().mul(100).round(1)}))
print('Unique tracks:', df.track_id.nunique())
print('Duplicate track IDs:', df.track_id.duplicated().sum())


## 2. Distribution of genres and key performance measures


In [ ]:
print(df.genre.value_counts())
display(df[['total_streams_mil','weeks_on_top_100','peak_chart_rank','danceability','energy']].describe().round(2))
fig, axes = plt.subplots(1, 2, figsize=(12,4))
df['genre'].value_counts().plot.bar(ax=axes[0], title='Track counts by genre')
df['total_streams_mil'].hist(bins=30, ax=axes[1])
axes[1].set(title='Lifetime streams distribution',xlabel='Millions of streams',ylabel='Tracks')
plt.tight_layout();plt.show()


## 3. Relationships relevant to a curator

A curator should not assume cumulative streams and chart longevity tell the same story. Both are historical, not live engagement.


In [ ]:
print('Pearson correlation:',round(df['weeks_on_top_100'].corr(df['total_streams_mil']),3))
fig, ax = plt.subplots(figsize=(7,4))
ax.scatter(df['weeks_on_top_100'],df['total_streams_mil'],alpha=.45)
ax.set(xlabel='Weeks in Top 100',ylabel='Lifetime streams (millions)',title='Historical chart longevity vs reach')
plt.tight_layout();plt.show()


## 4. EDA takeaway for the dashboard

- KPI cards describe the *currently eligible* shortlist, rather than unfiltered platform totals.
- The bar chart offers genre context; the scatter plot keeps chart longevity and reach separate.
- Genre, explicit-content policy, minimum chart weeks, and cumulative streams support meaningful editorial screening.
- Generic track/artist names and the absence of daily engagement metrics prevent claims about live performance or personalization.
